#  APIs (requests)

La forma más limpia de dar datos reales a un agente es usar una **API**: un servicio que ya devuelve la información lista para usar (el tiempo, el cambio de divisas, datos de países…).


## Configuración

In [ ]:
#!pip install -q requests

In [1]:
from openai import OpenAI
from getpass import getpass
import json

API_KEY = getpass("Pega aquí tu clave API: ")

BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "llama-3.3-70b-versatile"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)


In [ ]:
def ejecutar_agente(mensajes, tools, funciones, max_pasos=5, verbose=True):
    """Bucle de function calling: el modelo pide herramientas, nosotros las ejecutamos."""
    for paso in range(1, max_pasos + 1):
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=mensajes,
            tools=tools,            
            tool_choice="auto",     
            temperature=0,
        )
        msg = respuesta.choices[0].message

        
        if not msg.tool_calls:
            return msg.content


        mensajes.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            nombre = tc.function.name
            argumentos = json.loads(tc.function.arguments or "{}")
            funcion = funciones.get(nombre)
            if funcion is None:
                resultado = f"Error: la herramienta '{nombre}' no existe."
            else:
                try:
                    resultado = funcion(**argumentos)
                except Exception as e:
                    resultado = f"Error al ejecutar {nombre}: {e}"
            if verbose:
                print(f"[Paso {paso}] {nombre}({argumentos}) → {str(resultado)[:120]}")
            
            mensajes.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(resultado),
            })

    return "He alcanzado el límite de pasos."

## Primera llamada a una API con `requests`

Vamos a usar una API **gratuita y sin clave**: [Open-Meteo](https://open-meteo.com), que da el tiempo. Primero necesitamos las coordenadas de la ciudad (con su API de "geocoding") y luego el tiempo.

Una respuesta de API casi siempre viene en **JSON** (los mismos diccionarios de Python que ya conoces).

In [4]:
import requests

ciudad = "Vitoria-Gasteiz"
geo = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={"name": ciudad, "count": 1, "language": "es"},
).json()

lat = geo["results"][0]["latitude"]
lon = geo["results"][0]["longitude"]
print(f"{ciudad} está en {lat}, {lon}")


tiempo = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": lat, "longitude": lon, "current": "temperature_2m,wind_speed_10m"},
).json()

print("Temperatura ahora:", tiempo["current"]["temperature_2m"], "°C")
print("Viento:", tiempo["current"]["wind_speed_10m"], "km/h")

c:\Users\Diego Nuñez\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Vitoria-Gasteiz está en 42.84998, -2.67268
Temperatura ahora: 17.8 °C
Viento: 10.1 km/h


### Las piezas de una petición
- **URL**: la dirección del servicio.
- **params**: los datos que enviamos (la ciudad, las coordenadas…).
- **`.json()`**: convierte la respuesta en un diccionario de Python.

## Convertimos la API en una herramienta

Ahora empaquetamos esa lógica en una función limpia `consultar_tiempo(ciudad)` que el agente podrá usar.

In [5]:
def consultar_tiempo(ciudad):
    try:
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": ciudad, "count": 1, "language": "es"}, timeout=10).json()
        if not geo.get("results"):
            return f"No encuentro la ciudad '{ciudad}'."
        lat = geo["results"][0]["latitude"]; lon = geo["results"][0]["longitude"]
        t = requests.get("https://api.open-meteo.com/v1/forecast",
                         params={"latitude": lat, "longitude": lon, "current": "temperature_2m"},
                         timeout=10).json()
        return f"En {ciudad} hace {t['current']['temperature_2m']} °C."
    except Exception as e:
        return f"Error consultando el tiempo: {e}"

print(consultar_tiempo("Sevilla"))

En Sevilla hace 24.8 °C.


Añadimos una segunda API, también **sin clave**: [Frankfurter](https://frankfurter.dev), para el **cambio de divisas**.

In [6]:
def cambio_divisa(cantidad, desde, hacia):
    try:
        r = requests.get("https://api.frankfurter.app/latest",
                         params={"amount": cantidad, "from": desde, "to": hacia}, timeout=10).json()
        valor = r["rates"][hacia]
        return f"{cantidad} {desde} = {valor} {hacia}"
    except Exception as e:
        return f"Error con el cambio de divisa: {e}"

print(cambio_divisa(100, "EUR", "USD"))

100 EUR = 114.01 USD


In [8]:
mensajes = [{"role": "user", "content": "¿Qué tiempo hace en Vitoria-Gasteiz? Y cuántos dólares son 50 euros."}]


respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=mensajes,     
            temperature=0.3,
        )

print(respuesta.choices[0].message)

ChatCompletionMessage(content='Hasta la fecha de mi última actualización en abril de 2023, no puedo proporcionar información en tiempo real sobre el clima o los tipos de cambio actuales. Sin embargo, puedo ofrecerte algunas sugerencias sobre cómo obtener la información que necesitas.\n\nPara saber el tiempo en Vitoria-Gasteiz, España, te recomiendo consultar un sitio web de pronóstico del tiempo como AccuWeather, Weather.com, o el servicio meteorológico nacional de España, AEMET. Estos sitios suelen ofrecer pronósticos actualizados y detallados para diversas ubicaciones alrededor del mundo, incluyendo Vitoria-Gasteiz.\n\nEn cuanto a la conversión de euros a dólares, el tipo de cambio puede fluctuar constantemente debido a las condiciones del mercado. Para obtener el tipo de cambio más actualizado, te sugiero consultar un sitio web de finanzas como XE.com, Bloomberg, o Reuters, que ofrecen tipos de cambio en tiempo real. También puedes utilizar una aplicación de banca móvil o un servici

## Conectamos estas herramientas al agente

Reutilizamos `ejecutar_agente` del Notebook 1. Solo tenemos que describir las dos herramientas y registrarlas.

In [10]:
tools = [
    {"type": "function", "function": {
        "name": "consultar_tiempo",
        "description": "Devuelve el tiempo actual de una ciudad.",
        "parameters": {"type": "object",
                       "properties": {"ciudad": {"type": "string", "description": "Nombre de la ciudad"}},
                       "required": ["ciudad"]}}},
    {"type": "function", "function": {
        "name": "cambio_divisa",
        "description": "Convierte una cantidad de dinero de una moneda a otra (códigos como EUR, USD, GBP).",
        "parameters": {"type": "object",
                       "properties": {
                           "cantidad": {"type": "number", "description": "Cantidad a convertir"},
                           "desde": {"type": "string", "description": "Moneda de origen, p. ej. EUR"},
                           "hacia": {"type": "string", "description": "Moneda de destino, p. ej. USD"}},
                       "required": ["cantidad", "desde", "hacia"]}}},
]
funciones = {"consultar_tiempo": consultar_tiempo, "cambio_divisa": cambio_divisa}

mensajes = [{"role": "user", "content": "¿Qué tiempo hace en Madrid? Y cuántos dólares son 50 euros."}]
print(ejecutar_agente(mensajes, tools, funciones))

[Paso 1] consultar_tiempo({'ciudad': 'Madrid'}) → En Madrid hace 22.9 °C.
[Paso 1] cambio_divisa({'cantidad': 50, 'desde': 'EUR', 'hacia': 'USD'}) → 50 EUR = 57.005 USD
El tiempo en Madrid es de 22.9 °C. Y 50 euros son equivalentes a 57.005 dólares.


In [11]:
mensajes

[{'role': 'user',
  'content': '¿Qué tiempo hace en Madrid? Y cuántos dólares son 50 euros.'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': '0t62rfrdh',
    'type': 'function',
    'function': {'name': 'consultar_tiempo',
     'arguments': '{"ciudad":"Madrid"}'}},
   {'id': 'vth0a5jg6',
    'type': 'function',
    'function': {'name': 'cambio_divisa',
     'arguments': '{"cantidad":50,"desde":"EUR","hacia":"USD"}'}}]},
 {'role': 'tool',
  'tool_call_id': '0t62rfrdh',
  'content': 'En Madrid hace 22.9 °C.'},
 {'role': 'tool',
  'tool_call_id': 'vth0a5jg6',
  'content': '50 EUR = 57.005 USD'}]